In [ ]:
from openai import OpenAI
import os
import sys

api_key = os.environ.get("NVIDIA_API_KEY", "")
print(f"Using API Key: {api_key}")

_USE_COLOR = sys.stdout.isatty() and os.getenv("NO_COLOR") is None
_REASONING_COLOR = "\033[90m" if _USE_COLOR else ""
_RESET_COLOR = "\033[0m" if _USE_COLOR else ""

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = api_key
)


completion = client.chat.completions.create(
  model="z-ai/glm-5.1",
  messages=[{"role":"user","content":"Which number is larger, 9.11 or 9.8?"}],
  temperature=1,
  top_p=1,
  max_tokens=8192,
  extra_body={"chat_template_kwargs":{"enable_thinking":True,"clear_thinking":False}},
  stream=True
)

for chunk in completion:
  if not getattr(chunk, "choices", None):
    continue
  if len(chunk.choices) == 0 or getattr(chunk.choices[0], "delta", None) is None:
    continue
  delta = chunk.choices[0].delta
  reasoning = getattr(delta, "reasoning_content", None)
  if reasoning:
    print(f"{_REASONING_COLOR}{reasoning}{_RESET_COLOR}", end="")
  if getattr(delta, "content", None) is not None:
    print(delta.content, end="")

Using API Key: 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


In [ ]:
import os
import pandas as pd
from openai import OpenAI

# Setup client
api_key = os.environ.get("NVIDIA_API_KEY", "")
print(f"Using API Key: {api_key}")
client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = api_key
)

# Load dataset and get one sample
input_csv = "src/train.csv"
df = pd.read_csv(input_csv)
sample_row = df.iloc[0]

prompt_text = sample_row['prompt']
correct_answer = str(sample_row['answer']).strip()

print(f"--- PROMPT ---\n{prompt_text}\n")
print(f"--- EXPECTED ANSWER ---\n{correct_answer}\n")
print("--- MODEL OUTPUT ---")

system_prompt = "You are a helpful assistant. Please think step-by-step to solve the problem and provide your reasoning in English."
user_prompt = f"{prompt_text}\n\nPlease provide your final answer clearly."

completion = client.chat.completions.create(
  model="z-ai/glm-5.1",
  messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt}
  ],
  temperature=0.75,
  top_p=0.95,
  max_tokens=8192,
  extra_body={"chat_template_kwargs":{"enable_thinking":True,"clear_thinking":False}},
  stream=True
)

for chunk in completion:
    if not getattr(chunk, "choices", None):
        continue
    delta = chunk.choices[0].delta
    
    # Print reasoning
    r_text = getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None)
    if r_text:
        print(r_text, end="")
        
    # Print content
    c_text = delta.content
    if c_text is not None:
        print(c_text, end="")
        
print("\n\n--- DONE ---")

## MISTRAL LARGE-3


In [ ]:

import requests, base64

invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
stream = True


headers = {
  "Authorization": f"Bearer {os.environ.get('NVIDIA_API_KEY', '')}",
  "Accept": "text/event-stream" if stream else "application/json"
}

payload = {
  "model": "mistralai/mistral-large-3-675b-instruct-2512",
  "messages": [{"role":"user","content":"Hey??"}],
  "max_tokens": 2048,
  "temperature": 0.15,
  "top_p": 1.00,
  "frequency_penalty": 0.00,
  "presence_penalty": 0.00,
  "stream": stream
}

response = requests.post(invoke_url, headers=headers, json=payload)

if stream:
    for line in response.iter_lines():
        if line:
            print(line.decode("utf-8"))
else:
    print(response.json())


## DEEPSEEK V4-Pro


In [ ]:
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import time

# 1. Read API key
api_key = os.environ.get("NVIDIA_API_KEY", "")
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key
)

# 2. Load dataset
input_csv = "src/train.csv"
output_csv = "src/train_with_cot.csv"
df = pd.read_csv(input_csv)

# Resume support
if os.path.exists(output_csv):
    processed_df = pd.read_csv(output_csv)
    processed_ids = set(processed_df['id'].astype(str))
else:
    processed_ids = set()
    pd.DataFrame(columns=['id', 'prompt', 'answer', 'generated_cot']).to_csv(output_csv, index=False)

# 3. Process with 40 RPM pacing
for index, row in tqdm(df.iterrows(), total=len(df), desc="Generating CoT"):
    row_id = str(row['id'])
    if row_id in processed_ids:
        continue

    prompt_text = row['prompt']
    correct_answer = str(row['answer']).strip()

    # Prompt that includes the correct answer
    system_prompt = (
        "You are a helpful assistant. Given a problem and its correct answer, "
        "produce a detailed, step-by-step reasoning in English that leads to that answer. "
        "End your final response with the answer clearly stated."
    )
    user_prompt = (
        f"Problem: {prompt_text}\n"
        f"Correct answer: {correct_answer}\n\n"
        "Please write the reasoning that arrives at this answer."
    )

    success = False
    retries = 3
    while retries > 0:
        try:
            completion = client.chat.completions.create(
                model="deepseek-ai/deepseek-v4-pro",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.75,
                top_p=0.95,
                max_tokens=8192,
                extra_body={
                    "chat_template_kwargs": {
                        "thinking": True,
                        "reasoning_effort": "max"
                    }
                },
                stream=True
            )

            reasoning_pieces = []
            content_pieces = []

            for chunk in completion:
                if not getattr(chunk, "choices", None):
                    continue
                delta = chunk.choices[0].delta
                # Collect reasoning from thinking block (DeepSeek specific)
                r_text = getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None)
                if r_text:
                    reasoning_pieces.append(r_text)
                if delta.content:
                    content_pieces.append(delta.content)

            full_reasoning = "".join(reasoning_pieces)
            full_content = "".join(content_pieces)

            # Only save if we actually got reasoning and the answer appears
            if full_reasoning.strip() and correct_answer in full_content:
                new_row = pd.DataFrame([{
                    'id': row['id'],
                    'prompt': row['prompt'],
                    'answer': row['answer'],
                    'generated_cot': full_reasoning
                }])
                new_row.to_csv(output_csv, mode='a', header=False, index=False)
                processed_ids.add(row_id)
                success = True
            else:
                # Still log something? You may want to save empty or partial ones elsewhere.
                print(f"⚠️  CoT empty or missing answer for id {row_id}, skipping.")

            break  # exit retry loop on success

        except Exception as e:
            print(f"Error processing id {row_id}: {e}")
            retries -= 1
            if retries > 0:
                print(f"Retrying in 5 seconds...")
                time.sleep(5)
            else:
                print(f"❌ Failed after 3 retries for id {row_id}")

    # Pacing: 40 RPM => 1.5s between requests
    if success:
        time.sleep(1.5)
    else:
        # Even on failure, wait a bit to avoid hammering the API
        time.sleep(2)

In [ ]:
import os 
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = os.environ.get("NVIDIA_API_KEY", "")
)

print("Client initialized successfully.")
print(f"apikey: {client.api_key}")
completion = client.chat.completions.create(
  model="deepseek-ai/deepseek-v4-pro",
  messages=[{"role":"user","content":"""
You are a helpful assistant. Given a problem and its correct answer,
produce a detailed, step-by-step reasoning in English that leads to that answer.
End your final response with the answer clearly stated.       

In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. 
The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100 and the correct answer is 10010111"""}],
  temperature=1,
  top_p=0.95,
  max_tokens=8192,
  extra_body={"chat_template_kwargs":{"thinking":True,"reasoning_effort":"max"}},
  stream=True
)

for chunk in completion:
  if not getattr(chunk, "choices", None):
    continue
  reasoning = getattr(chunk.choices[0].delta, "reasoning", None) or getattr(chunk.choices[0].delta, "reasoning_content", None)
  if reasoning:
    print(reasoning, end="")
  if chunk.choices and chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

Client initialized successfully.
apikey: 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
We are given several input-output pairs for an 8-bit binary transformation. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions. We need to determine the output for 00110100, and the correct answer is 10010111.

Let's analyze the given pairs to deduce the transformation rule.

Given pairs:
Input -> Output
1) 01010001 -> 11011101
2) 00001001 -> 01101101
3) 00010101 -> 01010101
4) 11111111 -> 10000001
5) 10011101 -> 01000101
6) 00111011 -> 00001001
7) 10111101 -> 00000101
8) 00100110 -> 10110011

We need to find output for: 00110100 -> ?

All numbers are 8-bit. Let's write them as bits indexed from 7 to 0 (leftmost is bit 7, rightmost is bit 0). Or we can just work with strings.

We need to find a rule that applies to each input to produce the output. The rule might involve bitwise operations, shifts, r